In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

In [57]:
cv = pd.read_csv('./data/cv_control.csv', index_col=0).reset_index(drop=True)
print(cv.shape)
cv.head(2)

(107340, 15)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
0,000211c41b2c524ea758d997901733fe1720001995,5,0,-1,5,0,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-03 19:21:01.213,1,1,9100.0,0,1,0,0
1,000211c41b2c524ea758d997901733fe1720594007,8,2,1,8,22320,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-10 15:49:36.741,0,1,9100.0,0,1,0,0


In [58]:
sess = pd.DataFrame(cv.groupby('INCS_NO'), columns=['INCS_NO', 'SESS'])
sess.head(2)

,INCS_NO,SESS
0,00002708cacde8c67a97934afd9b5d592acde603b6d88c...,SESS_...
1,0003152c1436890b89d8593f863de552ef770f658de22a...,SESS...


In [113]:
sess_dv = []
dv = []
for i in range(sess.shape[0]):
    sess_len = sess['SESS'][i].shape[0]
    for j in range(sess_len):
        sess_id = sess['SESS'][i].iloc[j]['SESS_ID']
        sess['SESS'][i]['LST_SESS_TIME'] = pd.to_datetime(sess['SESS'][i]['LST_SESS_TIME'])
        if j == sess['SESS'][i].shape[0] - 1:
            sess_dv.append(sess_id)
            dv.append(0)
        elif sess['SESS'][i]['LST_SESS_TIME'].iloc[j+1] - sess['SESS'][i]['LST_SESS_TIME'].iloc[j] > pd.Timedelta(days=0.001):
            sess_dv.append(sess_id)
            dv.append(0)
        else:
            sess_dv.append(sess_id)
            dv.append(1)

In [114]:
sess_y = pd.DataFrame({'SESS_ID' : sess_dv,
                       'y' : dv})
sess_y['y'].value_counts()

y
0    104803
1      2537
Name: count, dtype: int64

In [115]:
temp = pd.merge(sess_y, cv, how='left', on='SESS_ID')
temp.head(2)

,SESS_ID,y,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
0,125A127FE89746BEBE4FCDC736833AB71720818297,0,13,4,1,13,118000,00002708cacde8c67a97934afd9b5d592acde603b6d88c...,2024-07-13 06:08:31.077,1,1,13050.0,1,0,0,0
1,125A127FE89746BEBE4FCDC736833AB71720941756,0,49,12,2,49,223320,00002708cacde8c67a97934afd9b5d592acde603b6d88c...,2024-07-14 16:34:58.583,0,1,13050.0,1,0,0,0


In [116]:
temp.drop(columns=['SESS_ID', 'INCS_NO', 'LST_SESS_TIME'], inplace=True)
temp.head(2)

,y,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
0,0,13,4,1,13,118000,1,1,13050.0,1,0,0,0
1,0,49,12,2,49,223320,0,1,13050.0,1,0,0,0


In [117]:
temp.to_csv('./control_sample.csv')